In [1]:
using Gridap
using GridapGmsh
using GridapEmbedded
using STLCutters

In [2]:
# Call Model
model_name = "3D_Plate_With_Hole.msh"
model_file = joinpath(@__DIR__,"..","Model_Creation", "Gmsh_Model", "Model", model_name)
isfile(model_file) || error("Path does not exist: $model_file")

true

In [3]:
model = GmshDiscreteModel(model_file)

Info    : Reading 'c:\Users\IIT BBSR\Desktop\Amiya\BTP\Poisson_Problem\..\Model_Creation\Gmsh_Model\Model\3D_Plate_With_Hole.msh'...
Info    : 53 entities
Info    : 2161 nodes
Info    : 10822 elements
Info    : Done reading 'c:\Users\IIT BBSR\Desktop\Amiya\BTP\Poisson_Problem\..\Model_Creation\Gmsh_Model\Model\3D_Plate_With_Hole.msh'


UnstructuredDiscreteModel()

In [4]:
# Call STL Meshing
mesh_name = "Box_Mesh.stl"
mesh_file = joinpath(@__DIR__, "..", "Model_Creation", "STL_Mesh", "Unfitted_Mesh", mesh_name)
isfile(mesh_file) || error("Path does not exist: $mesh_file")

true

In [5]:
geo = STLGeometry(mesh_file)

STLGeometry(Leaf((UnstructuredDiscreteModel(), "stl", nothing)))

In [ ]:
cutgeo = cut(model,geo)

In [ ]:
# Cell aggregation
aggregates = aggregate(AggregateAllCutCells(),cutgeo)

# Triangulations
Ω_act = Triangulation(cutgeo,ACTIVE)
Ω = Triangulation(cutgeo)
Γ = EmbeddedBoundary(cutgeo)
nΓ = get_normal_vector(Γ)   
dΩ = Measure(Ω,2)
dΓ = Measure(Γ,2)

# FE spaces
Vstd = TestFESpace(Ω_act,ReferenceFE(lagrangian,Float64,1))
V = AgFEMSpace(Vstd,aggregates)
U = TrialFESpace(V)

# Weak form
γ = 10.0
h = (pmax - pmin)[1] / cells[1]
ud(x) = x[1] - x[2]
f = 0
a(u,v) =
    ∫( ∇(v)⋅∇(u) )dΩ +
    ∫( (γ/h)*v*u  - v*(nΓ⋅∇(u)) - (nΓ⋅∇(v))*u )dΓ
l(v) =
    ∫( v*f )dΩ +
    ∫( (γ/h)*v*ud - (nΓ⋅∇(v))*ud )dΓ
# Solve
op = AffineFEOperator(a,l,U,V)
uh = solve(op)
writevtk(Ω,"results",cellfields=["uh"=>uh])

In [ ]:
const E = 210000.0                              # N/mm^2
const ν = 0.3                                   # Poissons Ratio

const μ = E/(2*(1+ν))
const λ = (E*ν)/((1+ν)*(1-2*ν))

In [ ]:
g = VectorValue(100.0, 0.0)                     # Neumann Boundary Condition
σ(ε) = λ*tr(ε)*one(ε) + 2*μ*ε                   # Sress Tensor

In [ ]:
# Γ_fixed   = tags = "D"
# Γ_Loaded = tags = "F"

In [ ]:
order = 1
reffe = ReferenceFE(lagrangian,VectorValue{2,Float64},order)
V0 = TestFESpace(model,reffe;
    conformity=:H1,
    dirichlet_tags=["Left"],
    dirichlet_masks=[(true,true)])
  
g1 = VectorValue(0.0,0.0)                           # Drichlet Boundary Condition
U = TrialFESpace(V0,[g1])  

In [ ]:
degree = 2*order
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)
Γ_load  = BoundaryTriangulation(model, tags = "Right")
dΓ_N = Measure(Γ_load,degree)

In [ ]:
# Weak from
# Internal work (Stiffness component)
a(u,v) = ∫( ε(v) ⊙ (σ∘ε(u)) )*dΩ 
# External Work
l(v) = ∫(v⋅g)*dΓ_N

In [ ]:
# Solve
op = AffineFEOperator(a,l,U,V0)
uh = solve(op)

In [ ]:
# create vtu file for checkup
result_path = joinpath(@__DIR__, "..", "..", "..", "Model", "FEM")
isdir(result_path) || mkpath(result_path)

In [ ]:
writevtk(Ω,joinpath(result_path, "Output"),
    cellfields=[
        "Displacement"=>uh,
        "Strain"=>ε(uh),
        "Stress"=>σ∘ε(uh)]
        )